## Smartphone Data Cleaning


### Objective

Identify, investigate, and resolve confirmed data-quality issues in the smartphone dataset.

### Data Cleaning Principles

- Preserve the original raw dataset without modification.
- Investigate missing values before deciding how to handle them.
- Correct only verified data errors and inconsistencies.
- Avoid removing valid records unless there is a clear justification.
- Document all data-cleaning decisions and changes.
- Validate the cleaned dataset to ensure data quality and consistency before saving it.


In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/smartphones.csv")

# creating a copy for working 
df_clean = df.copy()

print("Original Working Directory: ", df.shape)
print(" Working dataset shape: ", df_clean.shape)

Original Working Directory:  (980, 22)
 Working dataset shape:  (980, 22)


## 

### <b> 1. Missing-Value Investigation </b>
<b> Objective:</b> Identify which columns contain missing values and how frequently they occur.

<b> Decision rule:</b> Do not automatically fill or delete missing values. First investigate why they are missing and whether the information can be recovered.

In [290]:
df_clean.head()

,brand_name,model,price,avg_rating,5G_or_not,processor_brand,num_cores,processor_speed,battery_capacity,fast_charging_available,...,internal_memory,screen_size,refresh_rate,num_rear_cameras,os,primary_camera_rear,primary_camera_front,extended_memory_available,resolution_height,resolution_width
0,apple,Apple iPhone 11,38999,7.3,0,bionic,6.0,2.65,3110.0,0,...,64,6.1,60,2,ios,12.0,12.0,0,1792,828
1,apple,Apple iPhone 11 (128GB),46999,7.5,0,bionic,6.0,2.65,3110.0,0,...,128,6.1,60,2,ios,12.0,12.0,0,1792,828
2,apple,Apple iPhone 11 Pro Max,109900,7.7,0,bionic,6.0,2.65,3500.0,1,...,64,6.5,60,3,ios,12.0,12.0,0,2688,1242
3,apple,Apple iPhone 12,51999,7.4,1,bionic,6.0,3.10,NaN,0,...,64,6.1,60,2,ios,12.0,12.0,0,2532,1170
4,apple,Apple iPhone 12 (128GB),55999,7.5,1,bionic,6.0,3.10,NaN,0,...,128,6.1,60,2,ios,12.0,12.0,0,2532,1170


In [291]:
# so finding missing values in df_clean & which have > 0 that we are showing

missing_report = (
    df_clean.isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)
missing_report = missing_report[missing_report["missing_count"]> 0]
missing_report

,column,missing_count
3,avg_rating,101
5,processor_brand,20
6,num_cores,6
7,processor_speed,42
8,battery_capacity,11
10,fast_charging,211
16,os,14
18,primary_camera_front,5


In [292]:
# missing report percentage
missing_report["missing_percentage"] = (
    missing_report["missing_count"] / len(df_clean) * 100

).round(2)

missing_report = missing_report.sort_values(
    "missing_count", ascending=False
)
missing_report

,column,missing_count,missing_percentage
10,fast_charging,211,21.53
3,avg_rating,101,10.31
7,processor_speed,42,4.29
5,processor_brand,20,2.04
16,os,14,1.43
8,battery_capacity,11,1.12
6,num_cores,6,0.61
18,primary_camera_front,5,0.51


### <b> 2. Investigating Missing-Value Patterns </b>

<b>Objective:</b> Determine whether missing values in one column are associated with values in another column.

<b>Why:</b> Missing data may reflect unavailable specifications, incomplete records, or a relationship between features. We should investigate before choosing a cleaning strategy.

2.1 Check missing fast_charging values

In [293]:
charging_missing = df_clean["fast_charging"].isna()
print("Missing fast_charging values: ")
print(charging_missing.sum())

print("fast_charging_available distribution:")
print(df_clean["fast_charging_available"].value_counts(dropna=False))

print("\nCharging availabilty for missing fast_charging:")
print(
    df_clean.loc[
        charging_missing, "fast_charging_available"
    ].value_counts(dropna=False)
)


Missing fast_charging values: 
211
fast_charging_available distribution:
fast_charging_available
1    837
0    143
Name: count, dtype: int64

Charging availabilty for missing fast_charging:
fast_charging_available
0    143
1     68
Name: count, dtype: int64


so basis of fast_charging it showing fast charging_availble values 0 and 1 not entire 0 and 1

2.2 Check missing ratings by brand

In [294]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 980 entries, 0 to 979
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   brand_name                 980 non-null    str    
 1   model                      980 non-null    str    
 2   price                      980 non-null    int64  
 3   avg_rating                 879 non-null    float64
 4   5G_or_not                  980 non-null    int64  
 5   processor_brand            960 non-null    str    
 6   num_cores                  974 non-null    float64
 7   processor_speed            938 non-null    float64
 8   battery_capacity           969 non-null    float64
 9   fast_charging_available    980 non-null    int64  
 10  fast_charging              769 non-null    float64
 11  ram_capacity               980 non-null    int64  
 12  internal_memory            980 non-null    int64  
 13  screen_size                980 non-null    float64
 14  refre

In [295]:
rating_missing_by_brand = (
    df_clean.assign(
        rating_missing = df_clean["avg_rating"].isna()
    )
    .groupby("brand_name")
    .agg(
        total_phones = ("model","size"),
        missing_rating = ("rating_missing","sum")

    )
)
rating_missing_by_brand["missing_percntage"] = (
    rating_missing_by_brand["missing_rating"]
    /
    rating_missing_by_brand["total_phones"]
    * 100
).round(2)

rating_missing_by_brand.sort_values(
    "missing_rating", ascending=False
)

,total_phones,missing_rating,missing_percntage
brand_name,,,
samsung,132,16,12.12
vivo,111,15,13.51
xiaomi,134,9,6.72
itel,10,9,90.00
oppo,88,6,6.82
huawei,16,5,31.25
realme,97,5,5.15
oneplus,42,4,9.52
motorola,52,4,7.69


### 3 Missing-Value Pattern Analysis
<b>Objective:</b>Determine whether missing values occur together in the same rows.

<b>Why:</b>If multiple columns are missing in the same records, that may reveal a pattern that helps guide cleaning decisions.

In [296]:
missing_pattern = (
    df_clean.isna()
    .sum(axis=1)
    .value_counts()
)

print("Number of missing values per row:")
print(missing_pattern)

Number of missing values per row:
0    678
1    213
2     71
3     17
4      1
Name: count, dtype: int64


In [297]:
df_clean.isna().sum().sum()  # 213+(71*2)+(17*3)+(1*4)=410

np.int64(410)

In [298]:
missing_columns_together = (
    df_clean[df_clean.isna().any(axis=1)]
    .isna()
    .astype(int)
)

missing_columns_together

,brand_name,model,price,avg_rating,5G_or_not,processor_brand,num_cores,processor_speed,battery_capacity,fast_charging_available,...,internal_memory,screen_size,refresh_rate,num_rear_cameras,os,primary_camera_rear,primary_camera_front,extended_memory_available,resolution_height,resolution_width
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
901,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
902,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
903,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
906,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [299]:
row_missing_count = df_clean.isna().sum(axis=1)

rows_with_multiple_missing = df_clean.loc[
    row_missing_count >= 2
].copy()

rows_with_multiple_missing.insert(
    0, "missing_count",
    row_missing_count.loc[rows_with_multiple_missing.index]
)

rows_with_multiple_missing.sort_values(
    "missing_count", ascending=False
)

,missing_count,brand_name,model,price,avg_rating,5G_or_not,processor_brand,num_cores,processor_speed,battery_capacity,...,internal_memory,screen_size,refresh_rate,num_rear_cameras,os,primary_camera_rear,primary_camera_front,extended_memory_available,resolution_height,resolution_width
37,4,apple,Apple iPhone 15 Pro,130990,7.5,1,bionic,NaN,NaN,NaN,...,128,6.06,120,3,ios,50.0,12.0,0,2532,1170
45,3,apple,Apple iPhone XR2,71999,6.0,0,bionic,NaN,NaN,3060.0,...,64,6.10,60,2,ios,12.0,7.0,0,1792,828
43,3,apple,Apple iPhone SE 3 2022,43900,NaN,1,bionic,6.0,3.22,NaN,...,64,4.70,60,1,ios,12.0,7.0,0,1334,750
35,3,apple,Apple iPhone 15,82990,7.2,0,bionic,NaN,NaN,3285.0,...,128,6.06,60,2,ios,50.0,13.0,0,2532,1170
24,3,apple,Apple iPhone 14 Mini,69990,7.0,0,bionic,NaN,NaN,3500.0,...,128,5.42,60,2,ios,12.0,12.0,0,2340,1080
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
821,2,vivo,Vivo Y25,7499,NaN,0,helio,4.0,1.30,1900.0,...,16,4.50,60,1,android,5.0,2.0,1,854,480
831,2,vivo,Vivo Y55S,13490,NaN,0,snapdragon,4.0,1.20,2730.0,...,16,5.20,60,1,android,13.0,5.0,1,1280,720
834,2,vivo,Vivo Y71,25000,NaN,0,snapdragon,4.0,1.40,3360.0,...,16,6.00,60,1,android,13.0,5.0,1,1440,720
841,2,vivo,Vivo Y91c,8199,NaN,0,helio,8.0,2.00,4030.0,...,32,6.22,60,1,android,13.0,5.0,1,1520,720


### 4. Duplicate Record Investigation

<b>Objective</b>: Identify exact duplicate rows and repeated smartphone brand/model combinations.

<b>Why</b>: Duplicate records can distort statistics and future machine-learning results. However, repeated brand/model combinations are not automatically duplicates and must be investigated.

#### Check exact duplicate rows

In [300]:
exact_duplicates = df_clean.duplicated()
print("Exact Duplicate rows:",exact_duplicates.sum())

Exact Duplicate rows: 0


In [301]:
numeric_checks = {
    "price": df_clean["price"] <= 0,
    "battery_capacity": df_clean["battery_capacity"] <= 0,
    "processor_speed": df_clean["processor_speed"] <= 0,
    "avg_rating": (
        df_clean["avg_rating"].notna()
        & ~df_clean["avg_rating"].between(0, 10)
    ),
    "ram_capacity": df_clean["ram_capacity"] <= 0,
    "internal_memory": df_clean["internal_memory"] <= 0,
    "screen_size": df_clean["screen_size"] <= 0,
}

for column, condition in numeric_checks.items():
    print(f"{column}: {condition.fillna(False).sum()} suspicious values")

price: 0 suspicious values
battery_capacity: 0 suspicious values
processor_speed: 0 suspicious values
avg_rating: 0 suspicious values
ram_capacity: 0 suspicious values
internal_memory: 0 suspicious values
screen_size: 0 suspicious values


In [302]:
binary_columns = [
    "5G_or_not",
    "fast_charging_available",
    "extended_memory_available"
]

for column in binary_columns:
    invalid_values = df_clean.loc[
        df_clean[column].notna()
        & ~df_clean[column].isin([0, 1]),
        column
    ]

    print(f"\n{column}")
    print("Invalid values:", len(invalid_values))

    if not invalid_values.empty:
        print(invalid_values.value_counts())


5G_or_not
Invalid values: 0

fast_charging_available
Invalid values: 0

extended_memory_available
Invalid values: 0


In [303]:
cleaning_log = pd.DataFrame(
    columns=[
        "column",
        "issue",
        "decision",
        "reason"
    ]
)

cleaning_log

,column,issue,decision,reason


In [304]:
print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)

print("\nExact duplicate rows:", df_clean.duplicated().sum())

print("\nMissing values remaining:")
print(df_clean.isna().sum()[df_clean.isna().sum() > 0])

print("\nData types:")
print(df_clean.dtypes)

Original shape: (980, 22)
Cleaned shape: (980, 22)

Exact duplicate rows: 0

Missing values remaining:
avg_rating              101
processor_brand          20
num_cores                 6
processor_speed          42
battery_capacity         11
fast_charging           211
os                       14
primary_camera_front      5
dtype: int64

Data types:
brand_name                       str
model                            str
price                          int64
avg_rating                   float64
5G_or_not                      int64
processor_brand                  str
num_cores                    float64
processor_speed              float64
battery_capacity             float64
fast_charging_available        int64
fast_charging                float64
ram_capacity                   int64
internal_memory                int64
screen_size                  float64
refresh_rate                   int64
num_rear_cameras               int64
os                               str
primary_camera_rea

In [305]:
print("Original shape:",df.shape)
print("Working Shape:",df_clean.shape)

print("\nExact  duplicate rwos:")
print(df_clean.duplicated().sum())

print("\nMissing Cells:")
print(df_clean.isna().sum().sum())

print("\nMissing values by column:")
print(df_clean.isna().sum()[df_clean.isna().sum() > 0])

Original shape: (980, 22)
Working Shape: (980, 22)

Exact  duplicate rwos:
0

Missing Cells:
410

Missing values by column:
avg_rating              101
processor_brand          20
num_cores                 6
processor_speed          42
battery_capacity         11
fast_charging           211
os                       14
primary_camera_front      5
dtype: int64


In [306]:
cleaning_log = pd.DataFrame(
    columns=[
        "column",
        "issue",
        "decision",
        "reason"
    ]
)

cleaning_log

,column,issue,decision,reason


In [307]:
cleaning_log.loc[len(cleaning_log)] = {
    "column": "avg_rating",
    "issue": "101 missing values",
    "decision": "Retain as missing",
    "reason": "No reliable replacement ratings available"
}

cleaning_log


,column,issue,decision,reason
0,avg_rating,101 missing values,Retain as missing,No reliable replacement ratings available


In [308]:
numeric_missing_columns = [
    "num_cores",
    "processor_speed",
    "battery_capacity",
    "primary_camera_front"
]

df_clean[numeric_missing_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
num_cores,974.0,7.772074,0.836845,4.0,8.00,8.0,8.00,8.00
processor_speed,938.0,2.427217,0.464090,1.2,2.05,2.3,2.84,3.22
battery_capacity,969.0,4817.748194,1009.540054,1821.0,4500.00,5000.0,5000.00,22000.00
primary_camera_front,975.0,16.589744,10.876944,0.0,8.00,16.0,16.00,60.00


In [309]:
print("Highest battery capacities:")
print(
    df_clean.nlargest(10, "battery_capacity")[
        ["brand_name", "model", "battery_capacity"]
    ]
)

print("\nPhones with zero front camera:")
print(
    df_clean.loc[
        df_clean["primary_camera_front"] == 0,
        ["brand_name", "model", "primary_camera_front"]
    ]
)

Highest battery capacities:
    brand_name                           model  battery_capacity
58      doogee                    Doogee V Max           22000.0
409    oukitel                    Oukitel WP19           21000.0
410    oukitel                    Oukitel WP21            9800.0
411    oukitel                     Oukitel WP9            8000.0
624    samsung              Samsung Galaxy F63            7000.0
647    samsung              Samsung Galaxy M62            7000.0
711      tecno                    Tecno Pova 2            7000.0
712      tecno                    Tecno Pova 3            7000.0
713      tecno  Tecno Pova 3 (6GB RAM + 128GB)            7000.0
717      tecno                Tecno Pova Neo 2            7000.0

Phones with zero front camera:
    brand_name         model  primary_camera_front
172       itel  Itel A24 Pro                   0.0


In [310]:
df_clean.loc[
    df_clean["model"] == "Itel A24 Pro",
    [
        "brand_name",
        "model",
        "price",
        "primary_camera_front",
        "primary_camera_rear",
        "os"
    ]
]

,brand_name,model,price,primary_camera_front,primary_camera_rear,os
172,itel,Itel A24 Pro,5990,0.0,2.0,android


In [311]:
cleaning_log.loc[len(cleaning_log)] = {
    "column": "battery_capacity",
    "issue": "Maximum observed value is 22000 mAh",
    "decision": "Retain pending source verification",
    "reason": "High capacity alone does not establish that a value is invalid"
}
cleaning_log

,column,issue,decision,reason
0,avg_rating,101 missing values,Retain as missing,No reliable replacement ratings available
1,battery_capacity,Maximum observed value is 22000 mAh,Retain pending source verification,High capacity alone does not establish that a ...


### 5. Handling Missing Numeric Values

<b>Objective:</b> Handle missing numeric values using a documented strategy.

<b>Strategy:</b> Use median imputation as an initial approach for selected numeric features, provided it is appropriate for the feature and analysis.

Important: Keep the raw dataset unchanged and record all transformations.

#### A) Apply median imputation to the working copy

In [312]:
numeric_medians = {
    "num_cores": df_clean["num_cores"].median(),
    "processor_speed": df_clean["processor_speed"].median()
}

for column, median_value in numeric_medians.items():
    df_clean[column] = df_clean[column].fillna(median_value)

print("Missing values after imputation:")
print(df_clean[["num_cores", "processor_speed"]].isna().sum())

Missing values after imputation:
num_cores          0
processor_speed    0
dtype: int64


#### Record the decisions

In [313]:
for column, median_value in numeric_medians.items():
    cleaning_log.loc[len(cleaning_log)] = {
        "column": column,
        "issue": "Missing numeric values",
        "decision": f"Median imputation ({median_value})",
        "reason": "Preserve records while filling missing values with an observed central value"
    }
cleaning_log

,column,issue,decision,reason
0,avg_rating,101 missing values,Retain as missing,No reliable replacement ratings available
1,battery_capacity,Maximum observed value is 22000 mAh,Retain pending source verification,High capacity alone does not establish that a ...
2,num_cores,Missing numeric values,Median imputation (8.0),Preserve records while filling missing values ...
3,processor_speed,Missing numeric values,Median imputation (2.3),Preserve records while filling missing values ...


#### Handle battery_capacity

In [314]:
print("Missing battery capacity:", df_clean["battery_capacity"].isna().sum())
print("\nBattery capacity summary:")
print(df_clean["battery_capacity"].describe())

Missing battery capacity: 11

Battery capacity summary:
count      969.000000
mean      4817.748194
std       1009.540054
min       1821.000000
25%       4500.000000
50%       5000.000000
75%       5000.000000
max      22000.000000
Name: battery_capacity, dtype: float64


#### Handle primary_camera_front

In [315]:
print("Missing front-camera values:",
      df_clean["primary_camera_front"].isna().sum())

print("\nFront-camera summary:")
print(df_clean["primary_camera_front"].describe())

Missing front-camera values: 5

Front-camera summary:
count    975.000000
mean      16.589744
std       10.876944
min        0.000000
25%        8.000000
50%       16.000000
75%       16.000000
max       60.000000
Name: primary_camera_front, dtype: float64


In [316]:
# Cleaning decisions
# Column	Missing values	Median	Proposed action
# battery_capacity	11	5,000 mAh	Median imputation
# primary_camera_front	5	16 MP	Median imputation, after checking the meaning of zero

check the zero front-camera record

In [317]:
df_clean.loc[
    df_clean["primary_camera_front"] == 0,
    ["brand_name", "model", "primary_camera_front"]
]

,brand_name,model,primary_camera_front
172,itel,Itel A24 Pro,0.0


Fill missing battery capacities

In [318]:
battery_median = df_clean["battery_capacity"].median()

df_clean["battery_capacity"] = (
    df_clean["battery_capacity"].fillna(battery_median)
)

print("Missing battery capacity after imputation:",
      df_clean["battery_capacity"].isna().sum())

print("Imputation value:", battery_median)

Missing battery capacity after imputation: 0
Imputation value: 5000.0


In [319]:
k = df_clean["battery_capacity"] == 5000
k.isna().sum()

np.int64(0)

In [320]:
df_clean[df_clean["battery_capacity"] == 5000].head()

,brand_name,model,price,avg_rating,5G_or_not,processor_brand,num_cores,processor_speed,battery_capacity,fast_charging_available,...,internal_memory,screen_size,refresh_rate,num_rear_cameras,os,primary_camera_rear,primary_camera_front,extended_memory_available,resolution_height,resolution_width
3,apple,Apple iPhone 12,51999,7.4,1,bionic,6.0,3.1,5000.0,0,...,64,6.1,60,2,ios,12.0,12.0,0,2532,1170
4,apple,Apple iPhone 12 (128GB),55999,7.5,1,bionic,6.0,3.1,5000.0,0,...,128,6.1,60,2,ios,12.0,12.0,0,2532,1170
5,apple,Apple iPhone 12 (256GB),67999,7.6,1,bionic,6.0,3.1,5000.0,0,...,256,6.1,60,2,ios,12.0,12.0,0,2532,1170
6,apple,Apple iPhone 12 Mini,40999,7.4,1,bionic,6.0,3.1,5000.0,0,...,64,5.4,60,2,ios,12.0,12.0,0,2340,1080
7,apple,Apple iPhone 12 Mini (128GB),45999,7.5,1,bionic,6.0,3.1,5000.0,0,...,128,5.4,60,2,ios,12.0,12.0,0,2340,1080


Record the decision

In [321]:
cleaning_log.loc[len(cleaning_log)] = {
    "column": "battery_capacity",
    "issue": "11 missing values",
    "decision": f"Medium imputation({battery_median})",
    "reason": "Peserve smartphone records using the observed median"
}

In [322]:
cleaning_log

,column,issue,decision,reason
0,avg_rating,101 missing values,Retain as missing,No reliable replacement ratings available
1,battery_capacity,Maximum observed value is 22000 mAh,Retain pending source verification,High capacity alone does not establish that a ...
2,num_cores,Missing numeric values,Median imputation (8.0),Preserve records while filling missing values ...
3,processor_speed,Missing numeric values,Median imputation (2.3),Preserve records while filling missing values ...
4,battery_capacity,11 missing values,Medium imputation(5000.0),Peserve smartphone records using the observed ...


In [323]:
for column in ["processor_brand", "os"]:
    missing_before = df_clean[column].isna().sum()

    df_clean[column] = df_clean[column].fillna("Unknown")



In [324]:
cleaning_log.loc[len(cleaning_log)] = {
        "column": column,
        "issue": f"{missing_before} missing values",
        "decision": 'Replace missing values with "Unknown"',
        "reason": "Preserve records without inventing a category"
    }
cleaning_log

,column,issue,decision,reason
0,avg_rating,101 missing values,Retain as missing,No reliable replacement ratings available
1,battery_capacity,Maximum observed value is 22000 mAh,Retain pending source verification,High capacity alone does not establish that a ...
2,num_cores,Missing numeric values,Median imputation (8.0),Preserve records while filling missing values ...
3,processor_speed,Missing numeric values,Median imputation (2.3),Preserve records while filling missing values ...
4,battery_capacity,11 missing values,Medium imputation(5000.0),Peserve smartphone records using the observed ...
5,os,14 missing values,"Replace missing values with ""Unknown""",Preserve records without inventing a category


In [325]:
print("Final shape:", df_clean.shape)
print("Exact duplicates:", df_clean.duplicated().sum())
print("\nRemaining missing values:")
print(df_clean.isna().sum()[df_clean.isna().sum() > 0])
print("\nCleaning decisions:")
display(cleaning_log)

Final shape: (980, 22)
Exact duplicates: 0

Remaining missing values:
avg_rating              101
fast_charging           211
primary_camera_front      5
dtype: int64

Cleaning decisions:


,column,issue,decision,reason
0,avg_rating,101 missing values,Retain as missing,No reliable replacement ratings available
1,battery_capacity,Maximum observed value is 22000 mAh,Retain pending source verification,High capacity alone does not establish that a ...
2,num_cores,Missing numeric values,Median imputation (8.0),Preserve records while filling missing values ...
3,processor_speed,Missing numeric values,Median imputation (2.3),Preserve records while filling missing values ...
4,battery_capacity,11 missing values,Medium imputation(5000.0),Peserve smartphone records using the observed ...
5,os,14 missing values,"Replace missing values with ""Unknown""",Preserve records without inventing a category


In [326]:
df_clean["processor_brand"].isna().sum()

np.int64(0)

In [327]:
missing_report = (
    df_clean.isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)
missing_report = missing_report[missing_report["missing_count"]> 0]
missing_report

,column,missing_count
3,avg_rating,101
10,fast_charging,211
18,primary_camera_front,5


### final validation and save

In [328]:
print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)

Original shape: (980, 22)
Cleaned shape: (980, 22)


In [329]:
print("\nExact duplicate rows:", df_clean.duplicated().sum())


Exact duplicate rows: 0


In [330]:
print("\nMissing values remaining:")
remaining_missing = df_clean.isna().sum()
print(remaining_missing[remaining_missing > 0])


Missing values remaining:
avg_rating              101
fast_charging           211
primary_camera_front      5
dtype: int64


In [333]:
# Save processed dataset
from pathlib import Path

output_path = Path("../data/processed/smartphones_cleaned.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(output_path, index=False)

# Verify saved file
df_check = pd.read_csv(output_path)


### You Can see the previous data in df comparison with df_clean

In [243]:
missing_report = (
    df.isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)
missing_report = missing_report[missing_report["missing_count"]> 0]
missing_report

,column,missing_count
3,avg_rating,101
5,processor_brand,20
6,num_cores,6
7,processor_speed,42
8,battery_capacity,11
10,fast_charging,211
16,os,14
18,primary_camera_front,5


In [332]:
missing_report["missing_percentage"] = (
    missing_report["missing_count"] / len(df) * 100

).round(2)

missing_report = missing_report.sort_values(
    "missing_count", ascending=False
)
missing_report

,column,missing_count,missing_percentage
10,fast_charging,211,21.53
3,avg_rating,101,10.31
18,primary_camera_front,5,0.51


In [3]:
df_clean

,brand_name,model,price,avg_rating,5G_or_not,processor_brand,num_cores,processor_speed,battery_capacity,fast_charging_available,...,internal_memory,screen_size,refresh_rate,num_rear_cameras,os,primary_camera_rear,primary_camera_front,extended_memory_available,resolution_height,resolution_width
0,apple,Apple iPhone 11,38999,7.3,0,bionic,6.0,2.65,3110.0,0,...,64,6.10,60,2,ios,12.0,12.0,0,1792,828
1,apple,Apple iPhone 11 (128GB),46999,7.5,0,bionic,6.0,2.65,3110.0,0,...,128,6.10,60,2,ios,12.0,12.0,0,1792,828
2,apple,Apple iPhone 11 Pro Max,109900,7.7,0,bionic,6.0,2.65,3500.0,1,...,64,6.50,60,3,ios,12.0,12.0,0,2688,1242
3,apple,Apple iPhone 12,51999,7.4,1,bionic,6.0,3.10,NaN,0,...,64,6.10,60,2,ios,12.0,12.0,0,2532,1170
4,apple,Apple iPhone 12 (128GB),55999,7.5,1,bionic,6.0,3.10,NaN,0,...,128,6.10,60,2,ios,12.0,12.0,0,2532,1170
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
975,xiaomi,Xiaomi Redmi Note 9 Pro,13999,7.5,0,snapdragon,8.0,2.30,5020.0,1,...,64,6.67,60,4,android,48.0,16.0,1,2400,1080
976,xiaomi,Xiaomi Redmi Note 9 Pro (4GB RAM + 128GB),14439,7.7,0,snapdragon,8.0,2.30,5020.0,1,...,128,6.67,60,4,android,48.0,16.0,1,2400,1080
977,xiaomi,Xiaomi Redmi Note 9 Pro Max,16490,8.0,0,snapdragon,8.0,2.30,5020.0,1,...,64,6.67,60,4,android,64.0,32.0,1,2400,1080
978,zte,ZTE Axon 30S,19999,8.2,1,snapdragon,8.0,3.20,4200.0,1,...,128,6.90,120,4,android,50.0,16.0,1,2460,1080
